# Silver — Transformacion y Calidad de Datos
**LogiLake | D'LOGIA** — Capa Silver del Medallion Architecture

Operaciones en esta capa:
1. Casteo de tipos (strings -> timestamps, floats)
2. Columnas derivadas de logistica (dias de entrega, retraso, OTIF flag)
3. Validaciones de calidad de datos (DQ flags)
4. Deduplicacion por `order_id`
5. Escritura en Delta Lake Silver (UPSERT)

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("logilake_silver")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | Delta Lake activo")

26/04/04 02:12:26 WARN Utils: Your hostname, JuanCamiloDev resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/04 02:12:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/devcontainers/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/devcontainers/.ivy2/cache
The jars for the packages stored in: /home/devcontainers/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-16004ce2-90f3-48a5-adfa-63c201ec0cdd;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 166ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     

26/04/04 02:12:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.0 | Delta Lake activo


In [2]:
import os
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_PATH = "/mnt/c/logilake/data/bronze"
SILVER_PATH = "/mnt/c/logilake/data/silver/olist_orders"

os.makedirs(SILVER_PATH, exist_ok=True)
print(f"Bronze: {BRONZE_PATH}")
print(f"Silver: {SILVER_PATH}")

Bronze: /mnt/c/logilake/data/bronze
Silver: /mnt/c/logilake/data/silver/olist_orders


In [3]:
# Leer tablas Bronze (ingesta batch)
orders   = spark.read.format("delta").load(f"{BRONZE_PATH}/orders")
items    = spark.read.format("delta").load(f"{BRONZE_PATH}/order_items")
payments = spark.read.format("delta").load(f"{BRONZE_PATH}/order_payments")
reviews  = spark.read.format("delta").load(f"{BRONZE_PATH}/order_reviews")
products = spark.read.format("delta").load(f"{BRONZE_PATH}/products")
category = spark.read.format("delta").load(f"{BRONZE_PATH}/category_translation")
sellers  = spark.read.format("delta").load(f"{BRONZE_PATH}/sellers")

# Enriquecer items con categoria y estado del vendedor
products_r = products.join(
    category.select("product_category_name", "product_category_name_english"),
    "product_category_name", "left"
)
items_rich = (
    items
    .join(products_r.select("product_id", "product_category_name_english"), "product_id", "left")
    .join(sellers.select("seller_id", "seller_state"), "seller_id", "left")
)

items_agg = items_rich.groupBy("order_id").agg(
    F.count("order_item_id").cast("integer").alias("item_count"),
    F.sum("price").alias("total_items_value"),
    F.sum("freight_value").alias("total_freight"),
    F.collect_set("product_category_name_english").alias("categories"),
    F.collect_set("seller_state").alias("seller_states"),
)

payments_agg = payments.groupBy("order_id").agg(
    F.first("payment_type").alias("payment_type"),
    F.max("payment_installments").alias("payment_installments"),
    F.sum("payment_value").alias("payment_value"),
)

reviews_latest = (
    reviews.orderBy("review_creation_date")
    .dropDuplicates(["order_id"])
    .select("order_id", F.col("review_score").cast("float"))
)

# Renombrar columnas de orders al schema Silver
orders_r = orders.select(
    "order_id", "customer_id", "order_status",
    F.col("order_purchase_timestamp").alias("order_purchase_ts"),
    F.col("order_approved_at").alias("order_approved_ts"),
    F.col("order_delivered_customer_date").alias("order_delivered_ts"),
    F.col("order_estimated_delivery_date").alias("order_estimated_ts"),
    F.col("_bronze_ingested_at").alias("bronze_ingested_at"),
    F.col("_source").alias("source"),
)

bronze_df = (
    orders_r
    .join(items_agg, "order_id", "left")
    .join(payments_agg, "order_id", "left")
    .join(reviews_latest, "order_id", "left")
)

print(f"Bronze enriquecido: {bronze_df.count():,} filas")
bronze_df.printSchema()

26/04/04 02:12:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Bronze enriquecido: 99,441 filas
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_ts: timestamp (nullable = true)
 |-- order_approved_ts: timestamp (nullable = true)
 |-- order_delivered_ts: timestamp (nullable = true)
 |-- order_estimated_ts: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- total_items_value: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- seller_states: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- review_score: float (nullable = true)



In [4]:
# ── 2. Casteo de tipos + columnas derivadas de logistica ──────────────────────
TS_FORMAT = "yyyy-MM-dd HH:mm:ss"

silver_df = (
    bronze_df

    # Timestamps
    .withColumn("order_purchase_ts",
        F.to_timestamp(F.col("order_purchase_ts"), TS_FORMAT))
    .withColumn("order_approved_ts",
        F.to_timestamp(F.col("order_approved_ts"), TS_FORMAT))
    .withColumn("order_delivered_ts",
        F.to_timestamp(F.col("order_delivered_ts"), TS_FORMAT))
    .withColumn("order_estimated_ts",
        F.to_timestamp(F.col("order_estimated_ts"), TS_FORMAT))

    # Dias de entrega reales
    .withColumn("delivery_days_actual",
        F.datediff(
            F.col("order_delivered_ts"),
            F.col("order_purchase_ts")
        ).cast("integer"))

    # Dias de entrega estimados
    .withColumn("delivery_days_estimated",
        F.datediff(
            F.col("order_estimated_ts"),
            F.col("order_purchase_ts")
        ).cast("integer"))

    # Retraso (positivo = tardio, negativo = adelantado)
    .withColumn("delivery_delay_days",
        F.when(
            F.col("order_delivered_ts").isNotNull() & F.col("order_estimated_ts").isNotNull(),
            F.datediff(F.col("order_delivered_ts"), F.col("order_estimated_ts"))
        ).otherwise(F.lit(None)).cast("integer"))

    # Flags de estado
    .withColumn("is_delivered",
        F.col("order_status") == "delivered")
    .withColumn("is_canceled",
        F.col("order_status") == "canceled")
    .withColumn("is_late_delivery",
        F.when(
            F.col("is_delivered") & F.col("delivery_delay_days").isNotNull(),
            F.col("delivery_delay_days") > 0
        ).otherwise(F.lit(False)))

    # Valor total de la orden
    .withColumn("order_value_total",
        F.col("total_items_value").cast("double") +
        F.col("total_freight").cast("double"))

    # Ratio de flete
    .withColumn("freight_ratio",
        F.when(
            F.col("total_items_value").cast("double") > 0,
            F.col("total_freight").cast("double") /
            F.col("total_items_value").cast("double")
        ).otherwise(F.lit(None)))

    # Metadato Silver
    .withColumn("silver_processed_at", F.current_timestamp())
)

print("Columnas Silver:", len(silver_df.columns))
silver_df.printSchema()

Columnas Silver: 27
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_ts: timestamp (nullable = true)
 |-- order_approved_ts: timestamp (nullable = true)
 |-- order_delivered_ts: timestamp (nullable = true)
 |-- order_estimated_ts: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- source: string (nullable = true)
 |-- item_count: integer (nullable = true)
 |-- total_items_value: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- seller_states: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- review_score: float (nullable = true)
 |-- delivery_days_actual: integer (nullab

In [5]:
# ── 3. Validaciones de Calidad de Datos (DQ) ──────────────────────────────────
dq_flags_col = F.array(
    F.when(F.col("order_id").isNull(),
           F.lit("NULL_ORDER_ID")).otherwise(F.lit(None)),
    F.when(F.col("payment_value").isNull() | (F.col("payment_value") <= 0),
           F.lit("INVALID_PAYMENT_VALUE")).otherwise(F.lit(None)),
    F.when(F.col("item_count").isNull() | (F.col("item_count") < 1),
           F.lit("INVALID_ITEM_COUNT")).otherwise(F.lit(None)),
    F.when(
        F.col("order_delivered_ts").isNotNull() &
        F.col("order_purchase_ts").isNotNull() &
        (F.col("delivery_days_actual") < 0),
        F.lit("DELIVERY_BEFORE_PURCHASE")
    ).otherwise(F.lit(None)),
    F.when(
        F.col("review_score").isNotNull() &
        ((F.col("review_score") < 1) | (F.col("review_score") > 5)),
        F.lit("INVALID_REVIEW_SCORE")
    ).otherwise(F.lit(None)),
    F.when(
        F.col("delivery_days_actual").isNotNull() & (F.col("delivery_days_actual") > 120),
        F.lit("ANOMALOUS_DELIVERY_DAYS")
    ).otherwise(F.lit(None)),
)

silver_dq = (
    silver_df
    .withColumn("_dq_flags_arr", dq_flags_col)
    .withColumn("dq_flags",
        F.array_join(
            F.filter(F.col("_dq_flags_arr"), lambda x: x.isNotNull()),
            "|"
        )
    )
    .withColumn("dq_passed",
        F.size(F.filter(F.col("_dq_flags_arr"), lambda x: x.isNotNull())) == 0
    )
    .drop("_dq_flags_arr")
)

total  = silver_dq.count()
passed = silver_dq.filter(F.col("dq_passed")).count()
failed = total - passed
print(f"Total:   {total:,}")
print(f"DQ OK:   {passed:,} ({passed/total*100:.1f}%)")
print(f"DQ FAIL: {failed:,} ({failed/total*100:.1f}%)")

print("\nDistribucion de flags:")
silver_dq.filter(~F.col("dq_passed")) \
    .groupBy("dq_flags").count().orderBy(F.desc("count")).show()

Total:   99,441
DQ OK:   98,622 (99.2%)
DQ FAIL: 819 (0.8%)

Distribucion de flags:


+--------------------+-----+
|            dq_flags|count|
+--------------------+-----+
|  INVALID_ITEM_COUNT|  772|
|ANOMALOUS_DELIVER...|   43|
|INVALID_PAYMENT_V...|    3|
|INVALID_PAYMENT_V...|    1|
+--------------------+-----+



In [6]:
# ── 4. Deduplicacion por order_id ─────────────────────────────────────────────
window_spec = Window.partitionBy("order_id").orderBy(F.desc("bronze_ingested_at"))

silver_dedup = (
    silver_dq
    .withColumn("_row_num", F.row_number().over(window_spec))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

print(f"Antes dedup:   {silver_dq.count():,}")
print(f"Despues dedup: {silver_dedup.count():,}")

Antes dedup:   99,441


Despues dedup: 99,441


In [7]:
# ── 5. Escribir Silver (UPSERT con Delta MERGE) ───────────────────────────────
if not DeltaTable.isDeltaTable(spark, SILVER_PATH):
    silver_dedup.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .partitionBy("order_status") \
        .save(SILVER_PATH)
    print(f"Silver creado: {silver_dedup.count():,} filas")
else:
    silver_table = DeltaTable.forPath(spark, SILVER_PATH)
    (
        silver_table.alias("target")
        .merge(
            silver_dedup.alias("source"),
            "target.order_id = source.order_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("MERGE Silver completado")

26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.

26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% 

26/04/04 02:13:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers


26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/04 02:13:20 WARN MemoryManager: Total allocation exceeds 95.0

Silver creado: 99,441 filas


In [8]:
# ── 6. Verificacion Silver ────────────────────────────────────────────────────
silver_final = spark.read.format("delta").load(SILVER_PATH)
print(f"Silver total: {silver_final.count():,} filas\n")

print("KPI preview — metricas de logistica:")
silver_final.filter(F.col("is_delivered")).agg(
    F.round(F.avg("delivery_days_actual"), 1).alias("avg_delivery_days"),
    F.round(F.avg("delivery_days_estimated"), 1).alias("avg_estimated_days"),
    F.round(F.avg("delivery_delay_days"), 1).alias("avg_delay_days"),
    F.round(F.mean(F.col("is_late_delivery").cast("integer")) * 100, 1).alias("pct_late"),
    F.round(F.avg("payment_value"), 2).alias("avg_payment_value"),
    F.round(F.avg("review_score"), 2).alias("avg_review_score"),
).show(truncate=False)

print("\nDelta history:")
DeltaTable.forPath(spark, SILVER_PATH) \
    .history(3) \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(truncate=True)

Silver total: 99,441 filas

KPI preview — metricas de logistica:


+-----------------+------------------+--------------+--------+-----------------+----------------+
|avg_delivery_days|avg_estimated_days|avg_delay_days|pct_late|avg_payment_value|avg_review_score|
+-----------------+------------------+--------------+--------+-----------------+----------------+
|12.5             |24.4              |-11.9         |6.8     |159.86           |4.16            |
+-----------------+------------------+--------------+--------+-----------------+----------------+


Delta history:


+-------+--------------------+---------+--------------------+
|version|           timestamp|operation| operationParameters|
+-------+--------------------+---------+--------------------+
|      0|2026-04-04 02:13:...|    WRITE|{mode -> Overwrit...|
+-------+--------------------+---------+--------------------+

